# 中国货币政策报告文本特征与金融市场反应

本 Notebook 展示锁定分析计划、正式样本、章节修复、文本创新度、事件窗口、股票波动、股票收益和收益率曲线的核心计算。

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
while not (ROOT / 'configs/project.yml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
print(ROOT)

## 1. 锁定分析计划和正式样本

In [ ]:
from src.monetary_policy.sample import verify_final_analysis_plan, sample_bounds, is_in_formal_sample
verify_final_analysis_plan()
print('formal sample:', sample_bounds())

## 2. 数据来源和覆盖范围

In [ ]:
registry = pd.read_csv(ROOT / 'data/source_registry.csv')
registry[['dataset_name','source_organization','coverage_start','coverage_end','license_or_terms']].head()

In [ ]:
meta = pd.read_csv(ROOT / 'data/processed/pbc_report_metadata.csv')
meta['in_formal_sample'] = meta['report_period'].map(is_in_formal_sample)
meta.groupby('in_formal_sample').size()

## 3. 原始 PDF 抽取文本样例

In [ ]:
from src.monetary_policy.data.pbc_reports import report_text_path
sample_id = meta.loc[meta['in_formal_sample'], 'report_id'].iloc[-1]
sample_path = report_text_path(sample_id)
raw_text = sample_path.read_text(encoding='utf-8')
raw_text[:800]

## 4. 文本清洗和章节识别

In [ ]:
from src.monetary_policy.text.text_cleaner import normalize_text, split_sentences
cleaned = normalize_text(raw_text[:1200])
print(cleaned[:500])
print('sentences:', len(split_sentences(cleaned)))

## 5. 早期政策指引章节修复

In [ ]:
from src.monetary_policy.text.section_repair import repair_guidance_sections
repair_guidance_sections()
repair = pd.read_excel(ROOT / 'output/diagnostics/section_repair_report.xlsx')
repair

In [ ]:
sections = pd.read_csv(ROOT / 'data/processed/report_sections_repaired.csv')
sections[(sections['report_id'].isin(['2006Q1','2006Q4','2007Q2','2007Q4'])) & (sections['section']=='guidance')][['report_id','found','char_count','local_path']]

## 6. 中文分词和自定义政策短语

In [ ]:
from src.monetary_policy.text.tokenizer import tokenize
example = '保持流动性合理充裕，不搞大水漫灌，强化逆周期调节和跨周期调节。'
tokenize(example)

## 7. 金融情感词典和 PBC 领域词典

In [ ]:
from src.monetary_policy.text.lexicon import build_combined_lexicon, COMBINED_PATH
lexicon = build_combined_lexicon()
print(len(lexicon.positive), len(lexicon.negative), len(lexicon.dovish), len(lexicon.hawkish))
pd.read_csv(COMBINED_PATH).head()

## 8. 否定词和程度副词处理示例

In [ ]:
from src.monetary_policy.text.sentiment import score_text
for s in ['加大支持实体经济力度。','不搞大水漫灌。','更加有力支持稳增长。']:
    print(s, score_text(s, lexicon))

## 9. 文本指标计算

In [ ]:
from src.monetary_policy.pipeline import build_text_features
features = build_text_features()
features[['report_id','guidance_z_sentiment','macro_z_sentiment','guidance_z_policy_stance','guidance_unexpected_tone']].tail()

## 10. 扩展 TF-IDF 创新度

In [ ]:
features[['report_id','in_formal_sample','guidance_similarity_expanding_tfidf','guidance_novelty','fulltext_novelty_expanding_tfidf','similarity_char_ngram']].dropna(subset=['guidance_novelty']).tail()

## 11. 主题关注和未预期语调

In [ ]:
features.loc[features['in_formal_sample'], ['publication_datetime','guidance_z_sentiment','macro_z_sentiment','guidance_z_policy_stance','guidance_attention_growth','guidance_attention_inflation','guidance_unexpected_tone']].describe()

## 12. 人工标注完成后的文本验证

In [ ]:
from src.monetary_policy.text.manual_validation import load_filled_annotations, has_filled_annotations
print('人工标注已完成:', has_filled_annotations())
filled = load_filled_annotations()
print('标注句子数:', len(filled))
print('标注人:', filled['reviewer'].unique())
print()
print('情感标签分布:')
print(filled['manual_sentiment_label'].value_counts())
print()
print('政策倾向标签分布:')
print(filled['manual_policy_stance_label'].value_counts())
print()
print('主题标签分布:')
print(filled['manual_topic_label'].value_counts())

In [ ]:
from src.monetary_policy.text.validation_report import run_text_validation
vresult = run_text_validation()
print(f"情感 Accuracy: {vresult['summary']['sentiment_accuracy']:.4f}")
print(f"情感 Macro-F1: {vresult['summary']['sentiment_macro_f1']:.4f}")
print(f"政策倾向 Accuracy: {vresult['summary']['stance_accuracy']:.4f}")
print(f"政策倾向 Macro-F1: {vresult['summary']['stance_macro_f1']:.4f}")
print(f"主题 Accuracy: {vresult['summary']['topic_accuracy']:.4f}")
print(f"主题 Macro-F1: {vresult['summary']['topic_macro_f1']:.4f}")
print()
print('不一致句子总数:', vresult['summary']['disagreement_count'])
print('  情感不一致:', vresult['summary']['disagreement_sentiment_count'])
print('  政策倾向不一致:', vresult['summary']['disagreement_stance_count'])
print('  主题不一致:', vresult['summary']['disagreement_topic_count'])

## 13. 股票数据清洗

In [ ]:
stock = pd.read_csv(ROOT / 'data/processed/csi300_daily.csv', parse_dates=['date'])
stock[['date','close','simple_return','volatility_20d']].tail()

## 14. 债券收益率曲线数据

In [ ]:
bond = pd.read_csv(ROOT / 'data/processed/government_bond_yields.csv', parse_dates=['date'])
bond[['date','yield_1y','yield_5y','yield_10y','spread_10y_1y']].tail()

## 15. 发布时间和交易日对齐

In [ ]:
events = pd.read_csv(ROOT / 'data/processed/event_calendar.csv')
events['in_formal_sample'] = events['report_period'].map(is_in_formal_sample)
events['action_nearby_core'] = events['action_nearby']
events['action_nearby_extended'] = events.get('action_nearby_extended', events['action_nearby'])
events.loc[events['in_formal_sample'], ['event_id','publication_datetime','bond_event_date','equity_event_date','action_nearby_core','action_nearby_extended']].tail()

## 16. 修正后的窗口收益函数

In [ ]:
from src.monetary_policy.events.event_windows import window_return
prices = pd.Series([100, 102, 105, 110, 121])
print('0 to +3:', window_return(prices, 1, 0, 3))
print('-1 to +1:', window_return(prices, 1, -1, 1))

## 17. 事件面板

In [ ]:
from src.monetary_policy.events.event_panel import build_stock_event_panel, build_yield_curve_event_panel
stock_panel = build_stock_event_panel(features)
curve_daily, curve_panel = build_yield_curve_event_panel(features)
stock_panel[['event_id','return_0_p3','rv_0_5','log_rv_0_5','pre_event_volatility_20d']].tail()

## 18. 描述性统计

In [ ]:
stock_panel[['log_rv_0_5','guidance_novelty','guidance_novelty_x_post_2019','return_0_p3','guidance_z_sentiment']].describe()

## 19. 股票波动率主结果

In [ ]:
from src.monetary_policy.analysis.stock_volatility import run_stock_volatility_models
vol_table, main_vol, egarch = run_stock_volatility_models(stock_panel)
vol_table

In [ ]:
main_vol['params'], main_vol['pvalues'], main_vol['post_2019_total_effect'], main_vol['economic_effect']

## 20. 股票收益结果

In [ ]:
from src.monetary_policy.analysis.stock_returns import run_stock_return_models
return_table = run_stock_return_models(stock_panel)
return_table.head(12)

## 21. 收益率曲线水平、斜率和曲率

In [ ]:
curve_daily[['date','level','slope','curvature']].tail()

In [ ]:
from src.monetary_policy.analysis.yield_curve import run_yield_curve_models
yield_table = run_yield_curve_models(curve_panel)
yield_table

## 22. 原债券规格对照

In [ ]:
legacy = json.loads((ROOT / 'output/results/legacy_primary_result.json').read_text(encoding='utf-8')) if (ROOT / 'output/results/legacy_primary_result.json').exists() else json.loads((ROOT / 'output/results/primary/PRIMARY_RESULT_LOCK.json').read_text(encoding='utf-8'))
legacy['n'], legacy['params']['guidance_tone_change'], legacy['pvalues']['guidance_tone_change']

## 23. 稳健性检验和 Holm 校正

In [ ]:
from src.monetary_policy.analysis.robustness import similarity_robustness
similarity_robustness(stock_panel)

## 24. 图表源数据

In [ ]:
sorted([p.name for p in (ROOT / 'output/figures').glob('figure*.png')])

## 25. 结论和局限

文本创新度、股票收益和收益率曲线结果均由上述模块现场计算。解释时只讨论相关性，不把事件研究回归写成因果识别。

### 文本验证结论

人工标注已完成（标注人：罗允绩，240 句，涵盖政策指引和宏观章节）。验证结论如下：

- **金融情感**：自动词典的句子级准确率为 26.25%，主要问题是过度预测 positive（157/184 人工标注 neutral 的句子被自动分类为 positive）。原因在于姜富伟等和 Du et al. 的中文金融情感词典面向文档级金融市场分析设计，直接用于句子级政策文本时会产生系统性正向偏误。文档级聚合后的标准化指标在回归中更为稳健。
- **政策倾向**：自动词典的句子级准确率为 14.17%，hawkish 召回率为 0%。PBC 领域鹰鸽词典（v2 已扩展至 35+33 词）在句子级仍存在严重覆盖不足，原因是政策倾向常通过隐含、间接表达传递。文档级聚合可在一定程度上缓解这一问题。
- **主题分类**：v2 主题词典扩展后准确率从 38.33% 提升至 58.75%，growth 召回率从 35.1% 提升至 62.3%，financial_stability 召回率从 0% 提升至 36.8%。但 risk 和 inflation 的召回率仍偏低。

词典修订版本已保存在 `data/dictionaries/lexicon_versions/` 目录下，含 v1→v2 变更报告。